In [2]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder

In [21]:
daydf = pd.read_csv("../data/processed/processed_day.csv")

split = int(len(daydf) * 0.8)
train = daydf.iloc[:split]
test = daydf.iloc[split:]


change = ['season', 'mnth','weekday', 'weathersit']
encoder = OneHotEncoder(
    drop="first",
    sparse_output=False, handle_unknown="ignore")

train_encoded = encoder.fit_transform(train[change])
test_encoded = encoder.transform(test[change])

train_encoded = pd.DataFrame(
    train_encoded,
    columns=encoder.get_feature_names_out(change),
    index=train.index
)

test_encoded = pd.DataFrame(test_encoded, columns=encoder.get_feature_names_out(change), index = test.index)

train = train.drop(columns=change)
test = test.drop(columns=change)

train = pd.concat([train, train_encoded], axis=1)
test = pd.concat([test, test_encoded], axis=1)


features = [
    col for col in train.columns
    if col not in [
        "cnt",
        "casual",
        "registered",
        "dteday",
        "instant",
        "Unnamed: 0"
    ]
]
print(features)

target = "cnt"

X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

['yr', 'holiday', 'workingday', 'temp', 'atemp', 'hum', 'windspeed', 'cnt_lag1', 'cnt_lag7', 'cnt_roll7_mean', 'season_2', 'season_3', 'season_4', 'mnth_2', 'mnth_3', 'mnth_4', 'mnth_5', 'mnth_6', 'mnth_7', 'mnth_8', 'mnth_9', 'mnth_10', 'mnth_11', 'mnth_12', 'weekday_1', 'weekday_2', 'weekday_3', 'weekday_4', 'weekday_5', 'weekday_6', 'weathersit_2', 'weathersit_3']


In [22]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
lr_mae = mean_absolute_error(y_test, lr_pred)
lr_rmse = root_mean_squared_error(y_test, lr_pred)
lr_r2 = r2_score(y_test, lr_pred)
print(f"Linear regression MAE : {lr_mae:.2f}")
print(f"Linear regression RMSE: {lr_rmse:.2f}")
print(f"Linear regression R2: {lr_r2:.2f}")


Linear regression MAE : 659.12
Linear regression RMSE: 947.21
Linear regression R2: 0.75


In [23]:
rf_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = root_mean_squared_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)
print(f"Random forest MAE: {rf_mae:.2f}")
print(f"Random forest RMSE: {rf_rmse:.2f}")
print(f"Random forest R2: {rf_r2:.2f}")
importance = pd.Series(
    rf_model.feature_importances_,
    index=features
).sort_values(ascending=False)

print(importance)

Random forest MAE: 783.06
Random forest RMSE: 1035.08
Random forest R2: 0.70
cnt_roll7_mean    0.602521
cnt_lag1          0.152786
hum               0.060874
atemp             0.044492
yr                0.031609
temp              0.029955
windspeed         0.023973
cnt_lag7          0.013831
weathersit_3      0.013372
weathersit_2      0.006460
workingday        0.002779
weekday_6         0.002246
weekday_3         0.001718
mnth_4            0.001352
weekday_1         0.001329
season_2          0.001136
mnth_12           0.000867
weekday_5         0.000859
weekday_2         0.000820
mnth_3            0.000791
weekday_4         0.000740
holiday           0.000733
season_4          0.000720
mnth_5            0.000667
mnth_6            0.000657
mnth_10           0.000534
mnth_2            0.000523
season_3          0.000420
mnth_7            0.000365
mnth_11           0.000343
mnth_8            0.000315
mnth_9            0.000212
dtype: float64


In [24]:
for depth in [4, 6, 8, 10, None]:
    for leaf in [1, 5, 10]:
        m = RandomForestRegressor(n_estimators=200, max_depth=depth, min_samples_leaf=leaf, random_state=42, n_jobs=-1)
        m.fit(X_train, y_train)
        mae = mean_absolute_error(y_test, m.predict(X_test))
        print(f"depth={depth}, leaf={leaf}: MAE={mae:.2f}")

depth=4, leaf=1: MAE=869.81
depth=4, leaf=5: MAE=884.89
depth=4, leaf=10: MAE=902.06
depth=6, leaf=1: MAE=802.80
depth=6, leaf=5: MAE=820.82
depth=6, leaf=10: MAE=851.55
depth=8, leaf=1: MAE=788.33
depth=8, leaf=5: MAE=806.72
depth=8, leaf=10: MAE=847.94
depth=10, leaf=1: MAE=783.86
depth=10, leaf=5: MAE=805.18
depth=10, leaf=10: MAE=847.69
depth=None, leaf=1: MAE=783.06
depth=None, leaf=5: MAE=804.48
depth=None, leaf=10: MAE=847.74


In [25]:
gb_model = HistGradientBoostingRegressor()
gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)
gb_mae = mean_absolute_error(y_test, gb_pred)
gb_rmse = root_mean_squared_error(y_test, gb_pred)
gb_r2 = r2_score(y_test, gb_pred)
print(f"Hist Gradient Boosting MAE: {gb_mae:.2f}")
print(f"Hist Gradient Boosting RMSE: {gb_rmse:.2f}")
print(f"Hist Gradient Boosting R2: {gb_r2:.2f}")

Hist Gradient Boosting MAE: 850.74
Hist Gradient Boosting RMSE: 1084.67
Hist Gradient Boosting R2: 0.67
